# 03 · The preprocessing pipeline — build it yourself

Pairs with `docs/08` (Part A-B) and `docs/01-data-pipeline.md`. You'll rebuild the
whole `raw -> (N, 120)` transform and diff against `gpulab.data.preprocess`.

Runs without S3: we synthesize raw curves so you can experiment offline.

In [ ]:
import numpy as np
from scipy.signal import savgol_filter

def synthetic_raw(n_wells=16, n_frames=500, rng=None):
    """Falling-sigmoid melt curves whose derivative peaks in frames 380-460."""
    rng = rng or np.random.default_rng(0)
    t = np.arange(n_frames)
    rows = []
    for _ in range(n_wells):
        tm = rng.integers(390, 450)               # melt temperature (peak frame)
        curve = 1.0 / (1.0 + np.exp((t - tm) / 6.0))
        curve += rng.normal(0, 0.002, n_frames)   # noise
        rows.append(curve)
    return np.asarray(rows)

raw = synthetic_raw()
print("raw", raw.shape)

## 1. Smooth → negative gradient → smooth

The three-step core. Write it yourself with `savgol_filter(..., axis=1)` and
`-np.gradient(..., axis=1)`, then diff against the repo.

In [ ]:
def my_deriv(raw, w=13, p=2):
    # TODO: smooth (savgol, axis=1) -> -gradient(axis=1) -> smooth again
    ...

from gpulab.data.preprocess import smoothed_negative_derivative, PreprocessConfig
theirs = smoothed_negative_derivative(raw, PreprocessConfig())
# mine = my_deriv(raw)
# print("max abs diff:", np.abs(mine - theirs).max())   # want ~0

## 2. Peak-region ROI — fixed window (350-470), NOT aligned

We keep the peak at its true position so the melt temperature (Tm) stays a feature.
Slice frames 350:470 -> length 120.

In [ ]:
deriv = theirs
roi = deriv[:, 350:470]
print("roi", roi.shape, "| peak columns:", roi.argmax(axis=1)[:6])
# Note the peaks land at DIFFERENT columns -> Tm is preserved (that's the point).

## 3. AUC normalization (broadcasting recap)

Divide each curve by its own trapezoidal area so amplitude differences vanish but
shape and Tm remain.

In [ ]:
auc = np.trapezoid(roi, axis=1)
roi_norm = roi / auc[:, np.newaxis]
print("row areas after norm:", np.round(np.trapezoid(roi_norm, axis=1)[:4], 6))

In [ ]:
# Your turn: assemble my_preprocess(raw) = smooth/deriv -> [:,350:470] -> AUC,
# and diff against the repo's full pipeline:
from gpulab.data.preprocess import preprocess_curves
theirs_full = preprocess_curves(raw, PreprocessConfig())   # default = fixed window + AUC
print("repo output shape:", theirs_full.shape)             # (16, 120)
# TODO: mine = my_preprocess(raw); print(np.abs(mine - theirs_full).max())

## 4. Positive-well mining

A well is kept only if it has a real peak (`max(deriv[:,380:460]) > 4`) AND its
centered derivative never dips below `minimum_value`. Boolean masks in action.

In [ ]:
from gpulab.data.preprocess import positive_mask
mask = positive_mask(raw, PreprocessConfig(peak_threshold=0.0, minimum_value=-np.inf))
print("kept", mask.sum(), "of", len(mask), "wells")
# Your turn: reproduce the has_peak part yourself: deriv[:,380:460].max(axis=1) > thr
# TODO

> **Concepts to note** (copy into your own theory notebook):
> - Fixed unaligned window KEEPS Tm (peak position); peak-centering discards it.
> - Savitzky-Golay is a FIXED convolution kernel (see `savgol_coeffs`) -> this is
>   exactly what M11 reimplements as a GPU `conv1d`.
> - Mining = boolean masks + `flatnonzero`; preprocessing = smooth/deriv/crop/norm.
> - Self-check: which step would change if the instrument's frame count changed?